# Batch 평가 입력

Python 3.12 새 커널에서 세 사례를 Batch JSONL로 변환부터 확인합니다.

## Python 3.12 패키지 버전 고정

In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 세 사례를 Batch JSONL로 변환

In [2]:
# 세 평가 사례를 Responses endpoint용 Batch JSONL 행으로 변환해 저장합니다.
from pathlib import Path
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'labs/part4/ch15-evaluation-advanced').is_dir())
data = json.loads((root / 'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json').read_text())
out = root / 'labs/part4/ch15-evaluation-advanced/runs/batch-eval-input.jsonl'
rows = []
for case in data['cases']:
    rows.append({'custom_id': f"judge-{case['qid']}", 'method': 'POST',
      'url': '/v1/responses', 'body': {'model': 'gpt-5.4-mini',
      'input': f"질문: {case['question']}\n응답: {case['response']}"}})
out.parent.mkdir(exist_ok=True)
out.write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in rows))
print('저장:', out.name, '행:', len(rows))

저장: batch-eval-input.jsonl 행: 3


## 요청 형식과 요청 식별자 검증

In [3]:
# 각 JSONL 행의 식별자·method·endpoint·body 필드를 제출 전에 검사합니다.
loaded = [json.loads(line) for line in out.read_text().splitlines()]
ids = [row['custom_id'] for row in loaded]
checks = {
 'json_rows': len(loaded) == len(rows),
 'unique_custom_id': len(ids) == len(set(ids)),
 'post_only': all(x['method'] == 'POST' for x in loaded),
 'responses_endpoint': all(x['url'] == '/v1/responses' for x in loaded),
 'body_fields': all(x['body'].get('model') and x['body'].get('input') for x in loaded),
}
print(checks)
assert all(checks.values())

{'json_rows': True, 'unique_custom_id': True, 'post_only': True, 'responses_endpoint': True, 'body_fields': True}


## 모의 결과의 누락·중복·알 수 없는 ID 검사

In [4]:
# 로컬에서 실행한 범위와 실제 업로드·완료·청구 미확인 범위를 구분합니다.
mock_results = [
 {'custom_id': ids[0]}, {'custom_id': ids[1]},
 {'custom_id': ids[1]}, {'custom_id': 'judge-UNKNOWN'},
]
result_ids = [row['custom_id'] for row in mock_results]
missing = sorted(set(ids) - set(result_ids))
unknown = sorted(set(result_ids) - set(ids))
duplicates = sorted({value for value in result_ids if result_ids.count(value) > 1})
print({'누락': missing, '알 수 없는 ID': unknown, '중복': duplicates})
print('실제 업로드·완료·할인 청구: 미확인')
assert missing and unknown and duplicates

{'누락': ['judge-Q090'], '알 수 없는 ID': ['judge-UNKNOWN'], '중복': ['judge-Q006']}
실제 업로드·완료·할인 청구: 미확인
